In [1]:
"""
Linear Regression — implemented from scratch
Supports:
  • Closed-form solution  (Normal Equation)
  • Gradient Descent optimisation
"""

import math


# ── Utilities ─────────────────────────────────────────────────────────────────

def dot(a, b):
    return sum(x * y for x, y in zip(a, b))

def mat_vec_mul(M, v):
    """Matrix × vector."""
    return [dot(row, v) for row in M]

def transpose(M):
    return [[M[i][j] for i in range(len(M))] for j in range(len(M[0]))]

def mat_mul(A, B):
    BT = transpose(B)
    return [[dot(row_a, col_b) for col_b in BT] for row_a in A]

def add_bias_column(X):
    """Prepend a column of 1s for the intercept term."""
    return [[1.0] + list(row) for row in X]

def mse(y_true, y_pred):
    return sum((t - p) ** 2 for t, p in zip(y_true, y_pred)) / len(y_true)

def r_squared(y_true, y_pred):
    mean_y = sum(y_true) / len(y_true)
    ss_tot = sum((t - mean_y) ** 2 for t in y_true)
    ss_res = sum((t - p) ** 2 for t, p in zip(y_true, y_pred))
    return 1 - ss_res / ss_tot if ss_tot != 0 else 0.0


# ── Closed-Form (Normal Equation): θ = (XᵀX)⁻¹ Xᵀ y ─────────────────────────

def _invert_2x2(M):
    """Invert a 2×2 matrix (used for single-feature demo)."""
    a, b = M[0]
    c, d = M[1]
    det = a * d - b * c
    if abs(det) < 1e-12:
        raise ValueError("Matrix is singular; cannot invert.")
    return [[d / det, -b / det], [-c / det, a / det]]

def gaussian_elimination(A, b):
    """
    Solve Ax = b via Gaussian elimination with partial pivoting.
    Works for any square system — used for the normal equation.
    """
    n = len(b)
    # Augmented matrix [A | b]
    M = [row[:] + [b[i]] for i, row in enumerate(A)]

    for col in range(n):
        # Partial pivot
        max_row = max(range(col, n), key=lambda r: abs(M[r][col]))
        M[col], M[max_row] = M[max_row], M[col]

        pivot = M[col][col]
        if abs(pivot) < 1e-12:
            raise ValueError("Singular matrix encountered during elimination.")

        for row in range(col + 1, n):
            factor = M[row][col] / pivot
            M[row] = [M[row][k] - factor * M[col][k] for k in range(n + 1)]

    # Back-substitution
    x = [0.0] * n
    for i in range(n - 1, -1, -1):
        x[i] = (M[i][n] - sum(M[i][j] * x[j] for j in range(i + 1, n))) / M[i][i]
    return x


class LinearRegressionNormal:
    """Closed-form solution via the Normal Equation."""

    def __init__(self):
        self.theta = []   # [bias, w1, w2, …]

    def fit(self, X, y):
        Xb = add_bias_column(X)            # (m × n+1)
        XbT = transpose(Xb)                # (n+1 × m)
        XTX = mat_mul(XbT, Xb)            # (n+1 × n+1)
        XTy = mat_vec_mul(XbT, y)         # (n+1,)
        self.theta = gaussian_elimination(XTX, XTy)

    def predict(self, X):
        Xb = add_bias_column(X)
        return [dot(self.theta, row) for row in Xb]

    def score(self, X, y):
        preds = self.predict(X)
        return r_squared(y, preds), mse(y, preds)


# ── Gradient Descent ──────────────────────────────────────────────────────────

class LinearRegressionGD:
    """Gradient Descent optimisation."""

    def __init__(self, lr=0.01, epochs=1000, verbose=True):
        self.lr = lr
        self.epochs = epochs
        self.verbose = verbose
        self.theta = []
        self.loss_history = []

    def fit(self, X, y):
        Xb = add_bias_column(X)
        m = len(y)
        n = len(Xb[0])
        self.theta = [0.0] * n  # initialise weights to zero

        for epoch in range(1, self.epochs + 1):
            # Predictions
            preds = [dot(self.theta, row) for row in Xb]

            # Residuals
            errors = [p - t for p, t in zip(preds, y)]

            # Gradients: (1/m) Xᵀ errors
            grads = [
                sum(errors[i] * Xb[i][j] for i in range(m)) / m
                for j in range(n)
            ]

            # Parameter update
            self.theta = [t - self.lr * g for t, g in zip(self.theta, grads)]

            loss = sum(e ** 2 for e in errors) / (2 * m)
            self.loss_history.append(loss)

            if self.verbose and epoch % 200 == 0:
                print(f"  Epoch {epoch:5d} | MSE: {loss:.4f}")

    def predict(self, X):
        Xb = add_bias_column(X)
        return [dot(self.theta, row) for row in Xb]

    def score(self, X, y):
        preds = self.predict(X)
        return r_squared(y, preds), mse(y, preds)


# ── Demo ──────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    # y = 3x + 5  (with tiny noise)
    X_train = [[1], [2], [3], [4], [5], [6], [7], [8]]
    y_train = [8.1, 11.0, 14.2, 17.0, 20.1, 23.0, 26.1, 29.0]
    X_test  = [[9], [10]]
    y_test  = [32.0, 35.0]

    # --- Normal Equation ---
    print("=== Linear Regression — Normal Equation ===")
    model_ne = LinearRegressionNormal()
    model_ne.fit(X_train, y_train)
    r2, err = model_ne.score(X_test, y_test)
    print(f"  Learned params (bias, w): {[round(t, 4) for t in model_ne.theta]}")
    print(f"  Predictions on test: {[round(p, 2) for p in model_ne.predict(X_test)]}")
    print(f"  R²: {r2:.4f}  |  MSE: {err:.4f}\n")

    # --- Gradient Descent ---
    print("=== Linear Regression — Gradient Descent ===")
    model_gd = LinearRegressionGD(lr=0.01, epochs=1000)
    model_gd.fit(X_train, y_train)
    r2, err = model_gd.score(X_test, y_test)
    print(f"  Learned params (bias, w): {[round(t, 4) for t in model_gd.theta]}")
    print(f"  Predictions on test: {[round(p, 2) for p in model_gd.predict(X_test)]}")
    print(f"  R²: {r2:.4f}  |  MSE: {err:.4f}\n")

=== Linear Regression — Normal Equation ===
  Learned params (bias, w): [5.1, 2.9917]
  Predictions on test: [32.03, 35.02]
  R²: 0.9998  |  MSE: 0.0005

=== Linear Regression — Gradient Descent ===
  Epoch   200 | MSE: 0.9136
  Epoch   400 | MSE: 0.4120
  Epoch   600 | MSE: 0.1865
  Epoch   800 | MSE: 0.0851
  Epoch  1000 | MSE: 0.0395
  Learned params (bias, w): [4.4997, 3.0984]
  Predictions on test: [32.39, 35.48]
  R²: 0.9149  |  MSE: 0.1915

